# Capa Bronze — Ingesta de datos crudos
**Proyecto:** Detección de Fraude en Transacciones con Tarjeta de Crédito
**Objetivo de la capa:** Cargar el archivo fuente sin transformaciones de negocio, conservando todas las columnas originales, registrando metadatos de auditoría (fecha/hora de carga e identificación del archivo fuente) y persistiendo el resultado en formato Delta.

In [0]:
# 1. Configuración de catálogo y esquema
catalog = "fraud_proyecto"
schema_name = "bronze"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema_name}")

print(f"Catálogo y esquema listos: {catalog}.{schema_name}")

## 1. Lectura del archivo fuente (sin transformar)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# Ruta del archivo fuente en el Volume de Databricks
ruta_fraude = "/Volumes/fraud_proyecto/bronze/raw_data/input/csv/fraudTest.csv"
nombre_archivo_fuente = "fraudTest.csv"

# Lectura cruda: se conservan TODAS las columnas originales, sin renombrar ni castear
df_fraude_raw = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .option("sep", ",")
    .csv(ruta_fraude)
)

print(f"Columnas originales conservadas: {len(df_fraude_raw.columns)}")
df_fraude_raw.printSchema()

In [0]:
%pip install openpyxl

In [0]:
# CARGAR EXCEL
excel_path = "/Volumes/fraud_proyecto/bronze/raw_data/input/excel/fraudTest.xlsx"

import pandas as pd
from pyspark.sql.functions import lit, current_timestamp

# Leer con pandas y convertir a Spark DataFrame
pdf = pd.read_excel(excel_path)
df_excel = spark.createDataFrame(pdf)

# Agregar metadatos de auditoría
df_excel = df_excel.withColumn("formato_origen", lit("xlsx")) \
                   .withColumn("fecha_carga", current_timestamp()) \
                   .withColumn("archivo_origen", lit("fraudTest.xlsx"))

# Guardar como tabla Delta en Bronze
df_excel.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema_name}.fraud_test_xlsx")

print(f"Excel cargado: {df_excel.count()} registros")
df_excel.printSchema()

In [0]:
spark.sql(f"DESCRIBE TABLE {catalog}.{schema_name}.fraud_test_xlsx").show()

In [0]:
# carga del archivo JSON
ruta_json = "/Volumes/fraud_proyecto/bronze/raw_data/input/json/fraudTest.json"
nombre_archivo_json = "fraudTest.json"

df_json = (
    spark.read
    .option("multiline", "true")
    .json(ruta_json)
)

df_json = df_json.withColumn("formato_origen", lit("json")) \
                 .withColumn("fecha_carga", current_timestamp()) \
                 .withColumn("archivo_origen", lit(nombre_archivo_json))

# Guardar como tabla Delta
df_json.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalog}.{schema_name}.fraud_test_json")

print(f" JSON cargado: {df_json.count()} registros")
df_json.printSchema()

In [0]:
spark.sql(f"DESCRIBE TABLE {catalog}.{schema_name}.fraud_test_json").show()

## 2. Enriquecimiento con metadatos de auditoría (Bronze)

Se agregan columnas técnicas de trazabilidad sin alterar las columnas originales del archivo fuente para la capa Bronze:
- `fecha_hora_carga`: marca de tiempo del proceso de ingesta.
- `archivo_origen`: identificación del archivo fuente que generó el registro.

In [0]:
df_fraude = (
    df_fraude_raw
    .withColumn("fecha_hora_carga", current_timestamp())
    .withColumn("archivo_origen", lit(nombre_archivo_fuente))
)

display(df_fraude)

In [0]:
# Validación rápida de volumen de datos ingestados
total_registros = df_fraude.count()
print(f"Total de registros ingestados desde '{nombre_archivo_fuente}': {total_registros:,}")

## 3. Persistencia en formato Delta (capa Bronze)

In [0]:
(
    df_fraude.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(f"{catalog}.{schema_name}.fraud_test_original")
)

print(f"Tabla '{catalog}.{schema_name}.fraud_test_original' creada/actualizada correctamente.")

## 4. Verificación de la carga Bronze

In [0]:
display(
    spark.sql(f"""
        SELECT COUNT(*) AS total_registros,
               MIN(fecha_hora_carga) AS primera_carga,
               MAX(fecha_hora_carga) AS ultima_carga,
               COUNT(DISTINCT archivo_origen) AS archivos_fuente_distintos
        FROM {catalog}.{schema_name}.fraud_test_original
    """)
)